# EXEMPLO 1: Hybrid Query Básico

**Objetivo**: Referência rápida para implementar busca híbrida (BM25 + Vector) no OpenSearch

Este notebook demonstra:
- Setup mínimo para conexão
- Criação de índice híbrido em 5 linhas
- Pipeline RRF em 1 chamada
- Query híbrida completa
- Snippet copiável para reutilizar

## Setup Mínimo

In [ ]:
# Instalação
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opensearch-py'])

# Imports
from opensearchpy import OpenSearch
from typing import List, Dict, Any

# Conexão
client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False,
    verify_certs=False
)

print(f"✓ Conectado: {client.info()['version']['number']}")

## Criar Índice Híbrido (5 linhas)

In [ ]:
index_name = "indice_hibrido_exemplo1"

# Limpar índice anterior
try:
    client.indices.delete(index=index_name)
except:
    pass

# CRIAR ÍNDICE HÍBRIDO EM 5 LINHAS:
mapping = {
    "settings": {"number_of_shards": 1, "index": {"knn": True}},
    "mappings": {
        "properties": {
            "texto": {"type": "text"},
            "embedding": {"type": "knn_vector", "dimension": 384, "method": {"name": "hnsw", "space_type": "cosinesimil", "engine": "lucene"}}
        }
    }
}

client.indices.create(index=index_name, body=mapping)
print(f"✓ Índice criado: {index_name}")

## Inserir Documentos

In [ ]:
def simple_embedding(text: str, dim: int = 384) -> List[float]:
    """Gera embedding simples (hash-based). Use SentenceTransformer em produção."""
    import hashlib
    h = hashlib.md5(text.encode()).hexdigest()
    return [(int(h[i % len(h)], 16) % 100) / 100.0 for i in range(dim)]

# Dados de exemplo
docs = [
    {"id": "1", "texto": "Lei de Acesso à Informação garante direito a informações públicas"},
    {"id": "2", "texto": "Direitos fundamentais incluem vida, liberdade, igualdade e segurança"},
    {"id": "3", "texto": "LGPD protege dados pessoais e privacidade dos cidadãos"},
]

# Indexar
for doc in docs:
    doc["embedding"] = simple_embedding(doc["texto"])
    client.index(index=index_name, id=doc["id"], body={"texto": doc["texto"], "embedding": doc["embedding"]})

client.indices.refresh(index=index_name)
print(f"✓ {len(docs)} documentos indexados")

## Criar Pipeline RRF (1 chamada)

In [ ]:
# Deletar pipeline anterior se existir
try:
    client.transport.perform_request('DELETE', '/_search/pipeline/rrf_pipeline')
except:
    pass

# CRIAR PIPELINE RRF EM 1 CHAMADA:
pipeline_body = {
    "description": "Pipeline RRF para busca híbrida",
    "processors": [{"rrf": {"weights": [0.5, 0.5]}}]
}

client.transport.perform_request('PUT', '/_search/pipeline/rrf_pipeline', body=pipeline_body)
print("✓ Pipeline RRF criado")

## Query Híbrida Completa

In [ ]:
def hybrid_query(client, index: str, query_text: str, size: int = 5) -> List[Dict]:
    """
    Query híbrida:
    - multi_match: BM25 (busca por texto)
    - knn: Vector search (semântica)
    - RRF: Combina resultados
    """
    # Gerar embedding da query
    query_embedding = simple_embedding(query_text)
    
    # Query híbrida
    search_body = {
        "size": size,
        "query": {
            "bool": {
                "should": [
                    # BM25: busca por palavras-chave
                    {"multi_match": {"query": query_text, "fields": ["texto"]}},
                    # KNN: busca semântica
                    {"knn": {"embedding": {"vector": query_embedding, "k": size}}}
                ]
            }
        },
        # Usar pipeline RRF
        "ext": {"search_pipeline": "rrf_pipeline"}
    }
    
    response = client.search(index=index, body=search_body)
    
    # Processar resultados
    results = []
    for hit in response['hits']['hits']:
        results.append({
            'id': hit['_id'],
            'score': hit['_score'],
            'texto': hit['_source']['texto']
        })
    
    return results

# Executar query
query = "informação pública e privacidade"
resultados = hybrid_query(client, index_name, query, size=3)

print(f"\nQuery: '{query}'")
print(f"\nResultados ({len(resultados)}):")
for r in resultados:
    print(f"\n[{r['id']}] Score: {r['score']:.4f}")
    print(f"    {r['texto']}")

## Visualização dos Resultados

In [ ]:
import pandas as pd

# Converter para DataFrame
df = pd.DataFrame(resultados)
df = df[['id', 'score', 'texto']]

print("\nResultados em Tabela:")
print(df.to_string(index=False))
print(f"\nMelhor match score: {df['score'].max():.4f}")

## Snippet Copiável para Reutilizar

Cole este código em seus projetos para busca híbrida rápida:

In [ ]:
# ========== SNIPPET COPIÁVEL ==========
# Copie todo o código abaixo como template

snippet = '''
from opensearchpy import OpenSearch
from typing import List, Dict

# 1. Conectar
client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False
)

# 2. Função de embedding (use SentenceTransformer em produção)
def get_embedding(text: str, dim: int = 384) -> List[float]:
    import hashlib
    h = hashlib.md5(text.encode()).hexdigest()
    return [(int(h[i % len(h)], 16) % 100) / 100.0 for i in range(dim)]

# 3. Função de busca híbrida
def hybrid_search(query: str, index: str, size: int = 5):
    query_emb = get_embedding(query)
    body = {
        "size": size,
        "query": {
            "bool": {
                "should": [
                    {"multi_match": {"query": query, "fields": ["texto"]}},
                    {"knn": {"embedding": {"vector": query_emb, "k": size}}}
                ]
            }
        }
    }
    return client.search(index=index, body=body)

# 4. Usar
results = hybrid_search("sua query aqui", "seu_indice")
for hit in results["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Doc: {hit['_source']}")
'''

print(snippet)

## Resumo

**Tempo de implementação**: 5-10 minutos

**Componentes**:
- ✓ Índice híbrido com BM25 + KNN
- ✓ Pipeline RRF para rank fusion
- ✓ Query combinada
- ✓ Função reutilizável

**Próximos passos**:
- Integrar SentenceTransformer para embeddings reais
- Adicionar RAG pipeline para geração
- Implementar histórico conversacional